# 采样策略

In [1]:
from pathlib import Path

import requests

from src.core import *
from src.gpt import *

In [2]:
np.random.seed(42)

In [3]:
class GPTTrainerWithSampling(GPTTrainer):

    def generate(self, dataset, prompt, steps=256, temperature=1.0, top_k=None, top_p=None):
        self.model.eval()
        tokens = dataset.encode(prompt)

        with Tensor.no_grad():
            for _ in range(steps):
                feature = Tensor([tokens[-dataset.context_size:]])
                logits = self.model(feature)
                last_logits = logits.data[0, -1]

                if temperature == 0:
                    probs = np.zeros(len(last_logits))
                    probs[np.argmax(last_logits)] = 1.0
                else:
                    last_logits /= temperature

                    if top_k is not None:
                        threshold = np.partition(last_logits, -top_k)[-top_k]
                        last_logits = np.where(last_logits < threshold, -np.inf, last_logits)

                    if top_p is not None:
                        sorted_indices = np.argsort(last_logits)[::-1]
                        sorted_logits = last_logits[sorted_indices]
                        exp = np.exp(sorted_logits - np.max(sorted_logits))
                        cumulative_probs = np.cumsum(exp / np.sum(exp))
                        cutoff = np.searchsorted(cumulative_probs, top_p)
                        last_logits[sorted_indices[cutoff + 1:]] = -np.inf

                    exp = np.exp(last_logits - np.max(last_logits))
                    probs = exp / np.sum(exp)

                token = np.random.choice(len(probs), p=probs)
                tokens.append(token)

        return dataset.decode(tokens)

In [4]:
LEARNING_RATE = 0.001
BATCH_SIZE = 16
CONTEXT_SIZE = 256
EMBEDDING_SIZE = 384
HEADS = 6
BLOCKS = 6

In [10]:
DATA_FILE = "../../../tinyshakespeare.txt"

file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

In [11]:
MODEL_FILE = "../../../tinyshakespeare-gpt.npz"

dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
loss_fn = GPTLoss()
optimizer = AdamWOptimizer(model.parameters, lr=LEARNING_RATE)
trainer = GPTTrainerWithSampling(model, loss_fn, optimizer)
trainer.load(MODEL_FILE)

In [12]:
print(trainer.generate(dataset, prompt="ROMEO:", temperature=0.0))

ROMEO:
I will not be so much a man of the state,
And the senate of the sea true of the sea
That the senate of the sea that the senators,
And the sea that the senators of the sea
That the senators of the seasure of the state,
And the shall be straight of the seas


In [13]:
print(trainer.generate(dataset, prompt="ROMEO:", temperature=0.8))

ROMEO:
He shall happries a chart his sitful
kingly wife be some wounds or here.

POMPEY:
I would tell them to death, and we are in fair
greater.

GREGORY:
O grant with him.

AUFIDIUS:
A princed that damn'd it, as 'twas form,
When here before his abrace something


In [14]:
print(trainer.generate(dataset, prompt="ROMEO:", temperature=1.0, top_k=5))

ROMEO:
What's it the matter? Is't she had me to been,
As it i' that the mind mean and man the shadows,
And my brother, true me, to the world,
Is nothing tribunes that I would be say.
There I service the store of them tears.

SICINIUS:
I'll thee, a gall and threa


In [15]:
print(trainer.generate(dataset, prompt="ROMEO:", temperature=1.0, top_p=0.9))

ROMEO:
Ay, and of them not before.
They we fair will sudden to as all man?
Say, fame I take not, if an the mayor with thy cheek.

CLARENCE:
He made thy place.

GREGORY:
O help me, thou dost my report in thy countryment
The tale of Clarence of but friend, not hel


In [16]:
print(trainer.generate(dataset, prompt="ROMEO:", temperature=1.0, top_k=5, top_p=0.9))

ROMEO:
I saw not it is the soul both, with the strength.

CORIOLANUS:
I speak a sorrow word.

MARCIUS:
I hold him here, and I thence word want the stand of
The state of the parting of the word,
And when I say 'scape hath the before
That serves he will be suit to
